# Collaborative Filtering and Matrix Factorization — Exercises

Companion to the note [Collaborative Filtering and Matrix Factorization](Collaborative%20Filtering%20and%20Matrix%20Factorization.md).

Practise neighbourhood CF (cosine, Pearson, Jaccard, user- and item-based prediction), matrix factorization with SGD and ALS, and the implicit-feedback variants (weighted MF and BPR). By-hand exercises use a 5×5 rating matrix; code exercises use small synthetic low-rank data so that every model trains in a second.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×8 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import Ridge
nan = np.nan
# By-hand matrix: 5 users x 5 items, NaN = not rated
R5 = np.array([[5, 3, nan, 1, 4],
               [4, nan, nan, 1, 5],
               [1, 1, nan, 5, 2],
               [nan, 1, 5, 4, nan],
               [5, 4, 4, nan, 5]])

# Synthetic explicit data: 40 users x 30 items, rank-2 taste + biases + noise, ~50% observed, 80/20 split
_g = np.random.default_rng(0)
nu, ni = 40, 30
_full = np.clip(3.5 + _g.normal(0, .5, nu)[:, None] + _g.normal(0, .5, ni)[None, :]
                + 0.7 * _g.normal(size=(nu, 2)) @ _g.normal(size=(ni, 2)).T + 0.3 * _g.normal(size=(nu, ni)), 1, 5)
_u, _i = np.nonzero(_g.random((nu, ni)) < 0.5)
_perm = _g.permutation(len(_u)); _nt = int(0.8 * len(_u))
u_tr, i_tr, r_tr = _u[_perm[:_nt]], _i[_perm[:_nt]], _full[_u, _i][_perm[:_nt]]
u_te, i_te, r_te = _u[_perm[_nt:]], _i[_perm[_nt:]], _full[_u, _i][_perm[_nt:]]
R_tr = np.zeros((nu, ni)); R_tr[u_tr, i_tr] = r_tr          # 0 = missing (train only)
mask_tr = R_tr > 0

# Synthetic implicit data: 50 users x 40 items, one held-out positive per user
_g = np.random.default_rng(1)
nu_imp, ni_imp = 50, 40
X_imp = ((_g.normal(size=(nu_imp, 3)) @ _g.normal(size=(ni_imp, 3)).T + 0.5 * _g.normal(size=(nu_imp, ni_imp))) > 1.2).astype(float)
X_imp_tr = X_imp.copy(); held_out = np.full(nu_imp, -1)
for _uu in range(nu_imp):
    _pos = np.flatnonzero(X_imp[_uu])
    if len(_pos) >= 2:
        held_out[_uu] = _g.choice(_pos); X_imp_tr[_uu, held_out[_uu]] = 0

def heldout_auc(S):
    """Mean over users of P(score of held-out positive > score of a never-interacted item)."""
    a = []
    for u in range(nu_imp):
        if held_out[u] < 0:
            continue
        neg = np.flatnonzero(X_imp[u] == 0)
        a.append(np.mean(S[u, held_out[u]] > S[u, neg]))
    return float(np.mean(a))
rng = np.random.default_rng(42)

## Part A · Concepts

### Exercise 1 · User-based vs item-based
*🧠 Concept · ★☆☆*

Explain the difference between user-based and item-based neighbourhood CF. The note says item-based is "more stable".
Why? Think about a shop with 10M users and 100k items. Which similarities can be precomputed, and how often do they change?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare how many ratings an average item has vs an average user, and how fast a user's profile changes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**User-based:** find users similar to $u$ and average their (centred) ratings of item $i$. **Item-based:** find items similar to $i$
among those $u$ rated and average $u$'s own ratings of them.

Item-based is more stable because, with 10M users and 100k items, an average item has **100× more** ratings than an average user.
Item–item similarities are therefore estimated from many co-ratings and change slowly. They can be
precomputed offline (Amazon 2003). User profiles are short and change with every click, so user–user similarities are noisy and stale.
Item-based also gives explanations: "because you bought X".

</details>

### Exercise 2 · Why only the observed entries?
*🧠 Concept · ★★☆*

The MF objective sums only over observed pairs $\Omega$. What goes wrong if you instead fill missing ratings with 0 and take a
truncated SVD (or [[PCA]]) of the full matrix? How is the implicit-feedback WMF objective different, and why is summing over **all**
pairs acceptable there?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does a 0 mean on a 1–5 star scale? And what does a missing click mean in implicit data?

</details>

<details>
<summary><b>✅ Solution</b></summary>

With explicit ratings a 0 is not a rating: imputing 0 says "the user hates every unseen item". At 99% sparsity, SVD then mostly learns
to reproduce the zeros and predicts tiny values for everything. MF on $\Omega$ fits only real ratings and uses $\lambda$ to avoid overfitting.

In implicit WMF the target is the **preference** $p_{ui}\in\{0,1\}$, and an unobserved pair *is* weakly informative (probably
not interesting). Summing over all pairs with low confidence $c_{ui}=1$ for zeros and high confidence for observed ones encodes exactly that.
ALS makes the sum over all $|U|\cdot|I|$ pairs tractable via the trick $Q^\top C_uQ = Q^\top Q + Q^\top(C_u-I)Q$.

</details>

### Exercise 3 · SGD or ALS?
*🧠 Concept · ★☆☆*

Compare SGD and ALS for matrix factorization: per-step cost, parallelism, hyperparameters, and which one fits implicit WMF better. Why is each ALS
half-step a [[Linear Regression|ridge regression]]?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Fix $Q$. The loss as a function of a single $p_u$ is a sum of squared errors of a linear model in $p_u$ plus $\lambda\lVert p_u\rVert^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **SGD:** loops over observed ratings, cheap per step, easy to extend (biases, side features), but needs a learning rate and epochs, and is sequential.
- **ALS:** with $Q$ fixed, the loss decouples into independent ridge regressions $\min_{p_u}\sum_{i\in\Omega_u}(r_{ui}-p_u^\top q_i)^2+\lambda\lVert p_u\rVert^2$,
  solved in closed form $p_u=(Q_u^\top Q_u+\lambda I)^{-1}Q_u^\top r_u$. All users can be solved **in parallel**. There is no learning rate and the loss decreases monotonically.
- For implicit WMF the loss sums over **all** pairs, so there are far too many terms for SGD. ALS handles it with the $Q^\top Q$ precomputation trick, which is why iALS is the standard.

</details>

### Exercise 4 · Pointwise vs pairwise
*🧠 Concept · ★★☆*

BPR optimises $-\sum\ln\sigma(\hat x_{ui}-\hat x_{uj})$ over triples (user, observed $i$, unobserved $j$).
Why is a pairwise loss more natural than a pointwise squared error for top-K recommendation? Why does the note say it optimises **AUC**?
What role does negative sampling play?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

AUC = probability that a random positive is ranked above a random negative. Compare with $\sigma(\hat x_{ui}-\hat x_{uj})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Top-K only cares about the **order** of items for a user, not their absolute scores. A pairwise loss directly rewards ranking each
positive above each negative, while pointwise regression wastes capacity fitting exact values of 0/1 labels.

Per user, AUC $= \frac1{|I^+||I^-|}\sum_{i,j}\mathbb 1[\hat x_{ui}>\hat x_{uj}]$. BPR replaces the step function by the smooth
$\sigma(\hat x_{ui}-\hat x_{uj})$ and maximises its log, so it is a differentiable surrogate of AUC.

There are $|I^+|\cdot|I^-|$ pairs per user, far too many, so we **sample** a random unobserved $j$ per step. The sampling distribution
(uniform vs popularity) changes what the model learns.

</details>

## Part B · By hand

### Exercise 5 · Cosine similarity of two users
*✍️ By hand · ★☆☆*

In `R5`, users 0 and 1 have co-rated items 0, 3 and 4. Compute the cosine similarity of their rating vectors **over co-rated items only**.

In [ ]:
cos01 = None

_m = ~np.isnan(R5[0]) & ~np.isnan(R5[1])
check('cosine(u0, u1)', cos01, cosine_similarity(R5[0, _m][None], R5[1, _m][None])[0, 0])

<details>
<summary><b>💡 Hint</b></summary>

$u_0 = (5,1,4)$, $u_1 = (4,1,5)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Dot product $20+1+20=41$, both norms $\sqrt{42}$, so $\cos = 41/42 \approx 0.976$.
The similarity is very high even though user 0 gives item 0 a higher rating than user 1 does. Raw cosine ignores that users use the
scale differently, which is why Pearson centres the ratings first.

```python
cos01 = 41 / 42
```

</details>

### Exercise 6 · Pearson similarity
*✍️ By hand · ★☆☆*

Compute the Pearson correlation of users 0 and 1 over their co-rated items (centre each user by their mean **over the co-rated items**).

In [ ]:
pearson01 = None

_m = ~np.isnan(R5[0]) & ~np.isnan(R5[1])
check('Pearson(u0, u1)', pearson01, np.corrcoef(R5[0, _m], R5[1, _m])[0, 1])

<details>
<summary><b>💡 Hint 1</b></summary>

Both co-rated means are $10/3$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Centred: $u_0 = (5/3, -7/3, 2/3)$, $u_1 = (2/3, -7/3, 5/3)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Dot $= (10 + 49 + 10)/9 = 69/9$; squared norms $= (25+49+4)/9 = 78/9$ each.
$\rho = 69/78 \approx 0.885$. It is lower than the cosine because the common offset (both like everything at about 3.3) no longer counts as agreement.

```python
pearson01 = 69 / 78
```

</details>

### Exercise 7 · User-based prediction
*✍️ By hand · ★★☆*

Predict $\hat r_{u_1,\,i_1}$ (user 1, item 1) with the user-based formula from the note:
$$\hat r_{ui}=\bar r_u+\frac{\sum_{v\in N(u)}\mathrm{sim}(u,v)(r_{vi}-\bar r_v)}{\sum_v|\mathrm{sim}(u,v)|}.$$
Use the neighbours $N(u_1)=\{u_0,u_2,u_4\}$ with given similarities $\mathrm{sim}(u_1,u_0)=0.9$, $\mathrm{sim}(u_1,u_2)=-0.8$, $\mathrm{sim}(u_1,u_4)=0.6$
and user means over **all** their ratings in `R5`. Why does the negatively similar neighbour *raise* the prediction here?

In [ ]:
pred_u1_i1 = None

_mean = np.nanmean(R5, 1); _s = {0: 0.9, 2: -0.8, 4: 0.6}
_num = sum(s * (R5[v, 1] - _mean[v]) for v, s in _s.items()); _den = sum(abs(s) for s in _s.values())
check('prediction', pred_u1_i1, _mean[1] + _num / _den, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Means: $\bar r_0=3.25$, $\bar r_1=10/3$, $\bar r_2=2.25$, $\bar r_4=4.5$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Deviations on item 1: $u_0: 3-3.25=-0.25$, $u_2: 1-2.25=-1.25$, $u_4: 4-4.5=-0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Numerator: $0.9(-0.25) + (-0.8)(-1.25) + 0.6(-0.5) = -0.225 + 1.0 - 0.3 = 0.475$. Denominator $0.9+0.8+0.6=2.3$.
$\hat r = 10/3 + 0.475/2.3 \approx 3.333 + 0.207 = 3.540$.

User 2 has opposite taste and disliked item 1 ($-1.25$ below their mean). Multiplying by a negative similarity turns this into
evidence that user 1 will like it. In practice many systems keep only positive neighbours, because negative correlations are noisy.

```python
pred_u1_i1 = 10 / 3 + 0.475 / 2.3
```

</details>

### Exercise 8 · Jaccard for implicit data
*✍️ By hand · ★☆☆*

User A clicked items $\{1,2,3,5\}$ and user B clicked $\{2,3,4\}$. Compute the Jaccard similarity $|A\cap B|/|A\cup B|$.
Why is Jaccard used for implicit feedback instead of Pearson?

In [ ]:
jac_AB = None

from sklearn.metrics import jaccard_score
check('Jaccard', jac_AB, jaccard_score([0,1,1,1,0,1], [0,0,1,1,1,0]))

<details>
<summary><b>💡 Hint</b></summary>

Intersection $\{2,3\}$, union $\{1,2,3,4,5\}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$2/5 = 0.4$. Implicit data is binary (clicked or not), and there is no rating scale to centre. Jaccard compares the sets directly and
ignores the huge number of items neither user touched (shared zeros say nothing).

```python
jac_AB = 0.4
```

</details>

### Exercise 9 · One SGD step
*✍️ By hand · ★★☆*

Model $\hat r = p_u^\top q_i$ (no biases). Current values: $p_u=(0.5,\,0.2)$, $q_i=(0.4,\,0.1)$, observed $r_{ui}=4$, $\eta=0.1$, $\lambda=0.1$.
Perform one simultaneous SGD update from the note (both updates use the **old** vectors).

In [ ]:
p_new = None  # [., .]
q_new = None  # [., .]

_p = np.array([.5, .2]); _q = np.array([.4, .1]); _e = 4 - _p @ _q
check('p_u', p_new, _p + .1 * (_e * _q - .1 * _p))
check('q_i', q_new, _q + .1 * (_e * _p - .1 * _q))

<details>
<summary><b>💡 Hint</b></summary>

$\hat r = 0.2+0.02 = 0.22$, so $e = 3.78$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e = 4 - 0.22 = 3.78$.

$p_u \leftarrow (0.5,0.2) + 0.1\,[3.78(0.4,0.1) - 0.1(0.5,0.2)] = (0.5+0.1(1.512-0.05),\ 0.2+0.1(0.378-0.02)) = (0.6462,\ 0.2358)$.

$q_i \leftarrow (0.4,0.1) + 0.1\,[3.78(0.5,0.2) - 0.1(0.4,0.1)] = (0.4+0.1(1.89-0.04),\ 0.1+0.1(0.756-0.01)) = (0.585,\ 0.1746)$.

New prediction $\approx 0.6462\cdot0.585+0.2358\cdot0.1746 \approx 0.419$, which moved towards 4 as expected.

```python
p_new = [0.6462, 0.2358]
q_new = [0.585, 0.1746]
```

</details>

### Exercise 10 · An ALS half-step
*✍️ By hand · ★★☆*

Fix $Q$. A user rated three items with factors $q_1=(1,0)$, $q_2=(0,1)$, $q_3=(1,1)$ with ratings $r=(3,2,4)$. With $\lambda=1$, solve
$$p_u = (Q_u^\top Q_u + \lambda I)^{-1} Q_u^\top r_u$$
by hand (first derive it by setting the gradient of $\sum_i (r_{ui}-p_u^\top q_i)^2 + \lambda\lVert p_u\rVert^2$ to zero).

In [ ]:
p_als = None

_Q = np.array([[1, 0], [0, 1], [1, 1.]]); _r = np.array([3, 2, 4.])
check('p_u', p_als, Ridge(alpha=1.0, fit_intercept=False).fit(_Q, _r).coef_)

<details>
<summary><b>💡 Hint 1</b></summary>

$Q_u^\top Q_u = \begin{pmatrix}2&1\\1&2\end{pmatrix}$, $Q_u^\top r = (7, 6)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\begin{pmatrix}3&1\\1&3\end{pmatrix}^{-1} = \frac18\begin{pmatrix}3&-1\\-1&3\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Gradient: $-2Q_u^\top(r_u - Q_up_u) + 2\lambda p_u = 0 \Rightarrow (Q_u^\top Q_u + \lambda I)p_u = Q_u^\top r_u$.

$A = \begin{pmatrix}3&1\\1&3\end{pmatrix}$, $b=(7,6)$, $p_u = \frac18(21-6,\ -7+18) = (15/8,\ 11/8) = (1.875,\ 1.375)$.

```python
p_als = [1.875, 1.375]
```

</details>

### Exercise 11 · BPR loss and gradient
*✍️ By hand · ★★★*

With $\hat x_{ui} = p_u^\top q_i$ and $\hat x_{uij} = \hat x_{ui}-\hat x_{uj}$, the BPR loss for one triple is $\ell = -\ln\sigma(\hat x_{uij})$.

1. Show $\nabla_{p_u}\ell = -\sigma(-\hat x_{uij})(q_i - q_j)$, and give $\nabla_{q_i}\ell$ and $\nabla_{q_j}\ell$.
2. Evaluate $\ell$ and $\nabla_{p_u}\ell$ for $p_u=(1,0.5)$, $q_i=(0.5,0.5)$, $q_j=(0.2,1)$.

In [ ]:
bpr_loss = None
grad_pu = None

_p = np.array([1, .5]); _qi = np.array([.5, .5]); _qj = np.array([.2, 1.])
_f = lambda p: -np.log(expit(p @ (_qi - _qj)))
_eps = 1e-6
check('loss', bpr_loss, _f(_p), tol=1e-4)
check('grad wrt p_u (finite differences)', grad_pu, [(_f(_p + _eps*np.eye(2)[k]) - _f(_p - _eps*np.eye(2)[k])) / (2*_eps) for k in range(2)], tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

$\frac{d}{dx}\ln\sigma(x) = 1-\sigma(x) = \sigma(-x)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\hat x_{uij} = p_u^\top(q_i-q_j) = (1,0.5)\cdot(0.3,-0.5) = 0.05$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\nabla_\theta\ell = -\sigma(-\hat x_{uij})\,\nabla_\theta\hat x_{uij}$, so
$\nabla_{p_u}\ell = -\sigma(-\hat x_{uij})(q_i-q_j)$, $\nabla_{q_i}\ell = -\sigma(-\hat x_{uij})\,p_u$, $\nabla_{q_j}\ell = +\sigma(-\hat x_{uij})\,p_u$.
The factor $\sigma(-\hat x_{uij})$ is large when the pair is mis-ranked and close to 0 when it is already well ranked.

2. $\hat x_{uij}=0.05$, $\ell = -\ln\sigma(0.05) \approx 0.6685$. $\sigma(-0.05)\approx 0.4875$, so
$\nabla_{p_u}\ell \approx -0.4875\,(0.3,-0.5) = (-0.1463,\ 0.2438)$.

```python
_x = 0.05
bpr_loss = -np.log(expit(_x))
grad_pu = -expit(-_x) * np.array([0.3, -0.5])
```

</details>

### Exercise 12 · The factorization-machine trick
*✍️ By hand · ★★★*

A factorization machine models pairwise interactions $\sum_{i<j}\langle v_i,v_j\rangle x_ix_j$ of a feature vector $x\in\mathbb R^n$ with $v_i\in\mathbb R^k$.
Prove
$$\sum_{i<j}\langle v_i,v_j\rangle x_ix_j = \frac12\sum_{f=1}^k\Big[\Big(\sum_i v_{if}x_i\Big)^2 - \sum_i v_{if}^2x_i^2\Big],$$
and state the cost of each side. Then show that with $x$ = one-hot(user) ⊕ one-hot(item), the FM reduces to $p_u^\top q_i$ (MF).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\sum_{i<j} = \frac12(\sum_{i,j} - \sum_{i=j})$ and expand $\langle v_i,v_j\rangle = \sum_f v_{if}v_{jf}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\sum_{i<j}\langle v_i,v_j\rangle x_ix_j = \frac12\Big(\sum_{i}\sum_{j}\langle v_i,v_j\rangle x_ix_j - \sum_i\langle v_i,v_i\rangle x_i^2\Big)
= \frac12\sum_f\Big(\sum_i\sum_j v_{if}x_i\,v_{jf}x_j - \sum_i v_{if}^2x_i^2\Big) = \frac12\sum_f\Big[\big(\sum_i v_{if}x_i\big)^2 - \sum_i v_{if}^2x_i^2\Big].$

Left side: $O(kn^2)$. Right side: $O(kn)$, or even $O(k\cdot\mathrm{nnz}(x))$ for sparse $x$.

With $x$ having exactly two ones, at user $u$ and item $i$, the only nonzero pair is $(u,i)$, giving $\langle v_u, v_i\rangle = p_u^\top q_i$.
The linear terms $w_u + w_i$ are the biases $b_u+b_i$ and $w_0$ is $\mu$, so the FM is exactly biased MF. Extra features such as time or genre just add more ones.

</details>

## Part C · Code from scratch

### Exercise 13 · Item–item cosine similarity
*💻 Code · ★☆☆*

Write `item_cosine(R)` that takes a 0-filled rating matrix (users × items) and returns the items × items cosine similarity matrix,
using only NumPy. Apply it to the training matrix `R_tr`. Guard against items with no ratings (zero norm).

In [ ]:
def item_cosine(R):
    # TODO: normalise the columns, then one matrix product
    return None

S_items = item_cosine(R_tr)

check('item-item cosine', S_items, cosine_similarity(R_tr.T))

<details>
<summary><b>💡 Hint</b></summary>

`norms = np.linalg.norm(R, axis=0)`; `Rn = R / np.maximum(norms, 1e-12)`; `Rn.T @ Rn`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

One matrix product computes all $n_i^2$ similarities. On real data you would use sparse matrices and keep only the top-k neighbours per item.
Note that zero-filled cosine counts missing as 0. Mean-centring first ("adjusted cosine") is usually better for explicit ratings.

```python
def item_cosine(R):
    Rn = R / np.maximum(np.linalg.norm(R, axis=0), 1e-12)
    return Rn.T @ Rn

S_items = item_cosine(R_tr)
```

</details>

### Exercise 14 · Item-based kNN prediction
*💻 Code · ★★☆*

Implement `predict_item_knn(R, S, u, i, k)`: among the items **rated by user u**, take the `k` with highest similarity to `i`
(excluding `i` itself), and return
$$\hat r_{ui} = \frac{\sum_{j\in N_k(i;u)} s_{ij}\,r_{uj}}{\sum_{j}|s_{ij}|}$$
(fall back to the user's mean if the denominator is 0). Evaluate the RMSE on the test triples `(u_te, i_te, r_te)` with $k=5$
and store it in `rmse_knn`.

In [ ]:
def predict_item_knn(R, S, u, i, k=5):
    # TODO
    return None

rmse_knn = None

_S = cosine_similarity(R_tr.T)
def _ref_pred(u, i, k=5):
    cands = [(_S[i, j], R_tr[u, j]) for j in range(ni) if R_tr[u, j] > 0 and j != i]
    cands.sort(key=lambda t: -t[0]); cands = cands[:k]
    den = sum(abs(s) for s, _ in cands)
    return sum(s * r for s, r in cands) / den if den > 0 else R_tr[u][R_tr[u] > 0].mean()
check('single prediction', predict_item_knn(R_tr, _S, 3, 7, 5), _ref_pred(3, 7))
if rmse_knn is not None:
    check('test RMSE', rmse_knn, np.sqrt(np.mean([(_ref_pred(u, i) - r) ** 2 for u, i, r in zip(u_te, i_te, r_te)])))
else:
    check('test RMSE', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`rated = np.flatnonzero(R[u] > 0)`; drop `i`; `top = rated[np.argsort(-S[i, rated], kind='stable')[:k]]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The test RMSE comes out a bit above 1, which is worse than the bias baseline. Raw item-kNN on 0-filled cosine has no notion of user or item bias.
Practical item-kNN predicts $b_{ui} + \frac{\sum s_{ij}(r_{uj}-b_{uj})}{\sum|s_{ij}|}$ instead, i.e. it works on residuals of the baseline.

```python
def predict_item_knn(R, S, u, i, k=5):
    rated = np.flatnonzero(R[u] > 0)
    rated = rated[rated != i]
    top = rated[np.argsort(-S[i, rated], kind='stable')[:k]]
    den = np.abs(S[i, top]).sum()
    if den == 0:
        return R[u][R[u] > 0].mean()
    return S[i, top] @ R[u, top] / den

rmse_knn = np.sqrt(np.mean([(predict_item_knn(R_tr, S_items, u, i) - r) ** 2 for u, i, r in zip(u_te, i_te, r_te)]))
print('item-kNN test RMSE:', round(rmse_knn, 3))
```

</details>

### Exercise 15 · Biased MF with SGD
*💻 Code · ★★☆*

Implement the full model of the note,
$\hat r_{ui}=\mu+b_u+b_i+p_u^\top q_i$, trained with SGD on the training triples (`u_tr, i_tr, r_tr`):

$e=r-\hat r$; $b_u \mathrel{+}= \eta(e-\lambda b_u)$, $b_i \mathrel{+}= \eta(e-\lambda b_i)$, and the $p_u, q_i$ updates from the note.

Use $k=2$, $\eta=0.02$, $\lambda=0.05$, 60 epochs (shuffle each epoch), init $\mathcal N(0, 0.1^2)$. Store the test RMSE in `rmse_mf`.

In [ ]:
def train_mf(u_tr, i_tr, r_tr, k=2, lr=0.02, lam=0.05, epochs=60, seed=0):
    # TODO: return mu, bu, bi, P, Q
    return None

rmse_mf = None

_mu = r_tr.mean(); _bu = np.zeros(nu); _bi = np.zeros(ni)
for _ in range(10):   # reference: regularised bias-only baseline (alternating closed form)
    for j in range(ni):
        s = i_tr == j; _bi[j] = np.sum(r_tr[s] - _mu - _bu[u_tr[s]]) / (5 + s.sum())
    for v in range(nu):
        s = u_tr == v; _bu[v] = np.sum(r_tr[s] - _mu - _bi[i_tr[s]]) / (5 + s.sum())
_rmse_bias = np.sqrt(np.mean((_mu + _bu[u_te] + _bi[i_te] - r_te) ** 2))
print('bias-only baseline test RMSE:', round(_rmse_bias, 3))
if rmse_mf is not None:
    check('MF beats the bias baseline by a clear margin', rmse_mf < 0.8 * _rmse_bias, True)
else:
    check('MF beats the bias baseline', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Loop `for t in g.permutation(len(r_tr))`; compute `e` with the current parameters.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Update $p_u$ and $q_i$ simultaneously (tuple assignment) so both use the old values.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The bias-only baseline gets test RMSE ≈ 0.87, while MF reaches ≈ 0.39. The rank-2 interaction $p_u^\top q_i$ captures the taste
structure we planted in the data. Try $k=10$ with $\lambda=0$: training RMSE drops but test RMSE rises (overfitting); $\lambda$ fixes it.

```python
def train_mf(u_tr, i_tr, r_tr, k=2, lr=0.02, lam=0.05, epochs=60, seed=0):
    g = np.random.default_rng(seed)
    mu = r_tr.mean()
    P = 0.1 * g.normal(size=(nu, k)); Q = 0.1 * g.normal(size=(ni, k))
    bu = np.zeros(nu); bi = np.zeros(ni)
    for _ in range(epochs):
        for t in g.permutation(len(r_tr)):
            a, b, r = u_tr[t], i_tr[t], r_tr[t]
            e = r - (mu + bu[a] + bi[b] + P[a] @ Q[b])
            bu[a] += lr * (e - lam * bu[a]); bi[b] += lr * (e - lam * bi[b])
            P[a], Q[b] = P[a] + lr * (e * Q[b] - lam * P[a]), Q[b] + lr * (e * P[a] - lam * Q[b])
    return mu, bu, bi, P, Q

mu_hat, bu_, bi_, P_, Q_ = train_mf(u_tr, i_tr, r_tr)
pred = mu_hat + bu_[u_te] + bi_[i_te] + np.sum(P_[u_te] * Q_[i_te], 1)
rmse_mf = np.sqrt(np.mean((pred - r_te) ** 2))
print('MF test RMSE:', round(rmse_mf, 3))
```

</details>

### Exercise 16 · Alternating least squares
*💻 Code · ★★☆*

Implement `als_update(R, mask, F, lam)`: given fixed factors `F` of the *other* side (shape `n_cols × k`), return the matrix whose
row $a$ solves the ridge problem over the observed entries of row $a$ of `R`. Use it to alternate
`P = als_update(R_tr, mask_tr, Q, lam)` and `Q = als_update(R_tr.T, mask_tr.T, P, lam)` for 10 iterations ($k=2$, $\lambda=1$,
no biases). Record the regularised training loss after every iteration in `als_losses`.

In [ ]:
def als_update(R, mask, F, lam):
    # TODO: for each row a: solve (F_a^T F_a + lam I) x = F_a^T r_a over observed columns
    return None

als_losses = None

_g = np.random.default_rng(3); _F = _g.normal(size=(ni, 2))
_P = als_update(R_tr, mask_tr, _F, 1.0)
if _P is not None:
    _ref = np.array([Ridge(alpha=1.0, fit_intercept=False).fit(_F[mask_tr[a]], R_tr[a, mask_tr[a]]).coef_ for a in range(nu)])
    check('each row is a ridge regression', _P, _ref)
    check('ALS loss never increases', bool(np.all(np.diff(als_losses) <= 1e-9)), True)
else:
    check('ALS update', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`cols = np.flatnonzero(mask[a])`; `Fa = F[cols]`; `np.linalg.solve(Fa.T @ Fa + lam*np.eye(k), Fa.T @ R[a, cols])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each half-step solves its subproblem **exactly**, so the overall loss can only go down (block coordinate descent). No learning rate is needed.
The loss is not jointly convex in $(P,Q)$, though, so ALS reaches a local optimum that depends on the initialisation.

```python
def als_update(R, mask, F, lam):
    k = F.shape[1]
    out = np.zeros((R.shape[0], k))
    for a in range(R.shape[0]):
        cols = np.flatnonzero(mask[a])
        Fa = F[cols]
        out[a] = np.linalg.solve(Fa.T @ Fa + lam * np.eye(k), Fa.T @ R[a, cols])
    return out

def als_loss(P, Q, lam):
    E = (R_tr - P @ Q.T) * mask_tr
    return np.sum(E ** 2) + lam * (np.sum(P ** 2) + np.sum(Q ** 2))

g = np.random.default_rng(0)
P_als = 0.1 * g.normal(size=(nu, 2)); Q_als = 0.1 * g.normal(size=(ni, 2))
als_losses = []
for _ in range(10):
    P_als = als_update(R_tr, mask_tr, Q_als, 1.0)
    Q_als = als_update(R_tr.T, mask_tr.T, P_als, 1.0)
    als_losses.append(als_loss(P_als, Q_als, 1.0))
print(np.round(als_losses, 1))
```

</details>

### Exercise 17 · Implicit ALS: one weighted user update
*💻 Code · ★★★*

For weighted MF (Hu, Koren & Volinsky), the user update is
$$p_u = (Q^\top C_u Q + \lambda I)^{-1} Q^\top C_u\, \mathbf p(u),$$
with $C_u = \mathrm{diag}(c_{u1},\dots,c_{un})$, $c_{ui}=1+\alpha r_{ui}$ and $\mathbf p(u)$ the binary preferences, over **all** items.

1. Implement `ials_user(Q, r_u, alpha, lam)` directly.
2. Implement `ials_user_fast(Q, QtQ, r_u, alpha, lam)` using $Q^\top C_uQ = Q^\top Q + Q^\top(C_u-I)Q$, where $C_u - I$ is nonzero
   only on the observed items, so the cost per user depends on $|\Omega_u|$ instead of $n_{items}$.

Test on user 0 of `X_imp_tr` with random `Q` (40 × 4), α=10, λ=0.1.

In [ ]:
def ials_user(Q, r_u, alpha, lam):
    return None

def ials_user_fast(Q, QtQ, r_u, alpha, lam):
    return None

Q_rand = np.random.default_rng(5).normal(size=(ni_imp, 4))
p_slow = ials_user(Q_rand, X_imp_tr[0], 10, 0.1)
p_fast = ials_user_fast(Q_rand, Q_rand.T @ Q_rand, X_imp_tr[0], 10, 0.1)

_c = 1 + 10 * X_imp_tr[0]
_ref = Ridge(alpha=0.1, fit_intercept=False).fit(Q_rand, (X_imp_tr[0] > 0).astype(float), sample_weight=_c).coef_
check('weighted ridge (direct)', p_slow, _ref)
check('fast version agrees', p_fast, _ref)

<details>
<summary><b>💡 Hint 1</b></summary>

Direct: `C = np.diag(1 + alpha*r_u)`; solve `(Q.T@C@Q + lam*I) p = Q.T@C@pref`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Fast: `obs = np.flatnonzero(r_u > 0)`; `A = QtQ + (Q[obs].T * (alpha*r_u[obs])) @ Q[obs] + lam*I`; `b = Q[obs].T @ (1 + alpha*r_u[obs])` (since pref is 0 elsewhere).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The weighted problem is a ridge regression with sample weights $c_{ui}$, which is why sklearn's `Ridge(sample_weight=...)` reproduces it.
The fast version precomputes $Q^\top Q$ once per sweep ($O(n_ik^2)$) and then costs $O(|\Omega_u|k^2 + k^3)$ per user. This is what makes summing over **all** pairs feasible.

```python
def ials_user(Q, r_u, alpha, lam):
    C = np.diag(1 + alpha * r_u)
    pref = (r_u > 0).astype(float)
    return np.linalg.solve(Q.T @ C @ Q + lam * np.eye(Q.shape[1]), Q.T @ C @ pref)

def ials_user_fast(Q, QtQ, r_u, alpha, lam):
    obs = np.flatnonzero(r_u > 0)
    Qo = Q[obs]
    A = QtQ + (Qo.T * (alpha * r_u[obs])) @ Qo + lam * np.eye(Q.shape[1])
    b = Qo.T @ (1 + alpha * r_u[obs])
    return np.linalg.solve(A, b)

Q_rand = np.random.default_rng(5).normal(size=(ni_imp, 4))
p_slow = ials_user(Q_rand, X_imp_tr[0], 10, 0.1)
p_fast = ials_user_fast(Q_rand, Q_rand.T @ Q_rand, X_imp_tr[0], 10, 0.1)
```

</details>

### Exercise 18 · BPR-MF from scratch
*💻 Code · ★★★*

Train BPR-MF on the implicit matrix `X_imp_tr` with SGD: sample an observed pair $(u,i)$, sample an unobserved $j$ uniformly,
and update $p_u, q_i, q_j$ with your gradients from Exercise 11 plus L2 regularisation.
Use $k=8$, $\eta=0.05$, $\lambda=0.01$, 30 000 steps. Evaluate with `heldout_auc(P @ Q.T)` (provided in the setup) and compare with
popularity scores. Store the results in `auc_bpr` and `auc_pop`.

In [ ]:
def train_bpr(X, k=8, lr=0.05, lam=0.01, steps=30000, seed=0):
    # TODO: return P, Q
    return None

auc_bpr = None
auc_pop = None

if auc_bpr is not None:
    check('popularity AUC', auc_pop, heldout_auc(np.tile(X_imp_tr.sum(0), (nu_imp, 1))))
    check('BPR clearly beats popularity', auc_bpr > auc_pop + 0.15, True)
else:
    check('BPR', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

`us, its = np.nonzero(X)`; each step pick `t = g.integers(len(us))`, then resample `j` until `X[u, j] == 0`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`s = expit(-(P[u] @ (Q[i] - Q[j])))`; `P[u] += lr*(s*(Q[i]-Q[j]) - lam*P[u])`; `Q[i] += lr*(s*pu - lam*Q[i])`; `Q[j] += lr*(-s*pu - lam*Q[j])` with `pu` the old `P[u]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Popularity reaches AUC ≈ 0.57 here, and BPR ≈ 0.89. The held-out positive is ranked above about 89% of a user's unseen items.
(On real data popularity is much stronger than on this synthetic set, which has no planted popularity effect.)

```python
def train_bpr(X, k=8, lr=0.05, lam=0.01, steps=30000, seed=0):
    g = np.random.default_rng(seed)
    n_u, n_i = X.shape
    P = 0.1 * g.normal(size=(n_u, k)); Q = 0.1 * g.normal(size=(n_i, k))
    us, its = np.nonzero(X)
    for _ in range(steps):
        t = g.integers(len(us)); u, i = us[t], its[t]
        j = g.integers(n_i)
        while X[u, j]:
            j = g.integers(n_i)
        s = expit(-(P[u] @ (Q[i] - Q[j])))
        pu = P[u].copy()
        P[u] += lr * (s * (Q[i] - Q[j]) - lam * P[u])
        Q[i] += lr * (s * pu - lam * Q[i])
        Q[j] += lr * (-s * pu - lam * Q[j])
    return P, Q

P_b, Q_b = train_bpr(X_imp_tr)
auc_bpr = heldout_auc(P_b @ Q_b.T)
auc_pop = heldout_auc(np.tile(X_imp_tr.sum(0), (nu_imp, 1)))
print('BPR AUC', round(auc_bpr, 3), '| popularity AUC', round(auc_pop, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Collaborative Filtering and Matrix Factorization](Collaborative%20Filtering%20and%20Matrix%20Factorization.md).*